# CoDE-SAT Demo: ConvNeXt vs. DeiT auf EuroSAT

**Voraussetzungen:**
- Checkpoints in `outputs/checkpoints/` (`*_sgd_stage1_best.pt`, alle drei Modelle)
- EuroSAT-Daten vorhanden (`scripts/prepare_eurosat.py` + `scripts/make_splits.py` gelaufen)
- eigene Fotos (JPG/PNG) im Ordner `demo_images/`

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import timm
import torch
from PIL import Image
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
    "ConvNeXt-Tiny": "convnext_tiny.fb_in1k",
    "DeiT-Small": "deit_small_patch16_224.fb_in1k",
    "DeiT-Small dist.": "deit_small_distilled_patch16_224.fb_in1k",
}
CKPT_DIR = Path("outputs/checkpoints")
SPLITS = Path("data/splits")

classes = (
    pd.read_csv(SPLITS / "classes.txt").sort_values("label")["class_name"].tolist()
)

# same transforms as during training (resize 224 + imagenet normalization)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"device: {DEVICE}")
print(f"{len(classes)} Klassen: {classes}")

In [ ]:
models = {}

for name, timm_name in MODELS.items():
    # pretrained=False, the weights come entirely from our checkpoint
    model = timm.create_model(timm_name, pretrained=False, num_classes=len(classes))

    ckpt_path = CKPT_DIR / f"{timm_name}_sgd_stage1_best.pt"
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    models[name] = model.to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"{name}: geladen ({n_params:.1f}M Parameter)")

In [ ]:
@torch.no_grad()
def predict_top3(model, image):
    """top-3 classes with softmax confidence for a PIL image"""
    x = transform(image).unsqueeze(0).to(DEVICE)
    probs = model(x).softmax(dim=1)[0]
    top = probs.argsort(descending=True)[:3]
    return [(classes[i], probs[i].item()) for i in top]

## Teil 1: Drei zufällige Testbilder

Jedes Ausführen der Zelle zieht neue Bilder — gut für die Live-Demo.
(✓/✗ = Top-1-Vorhersage richtig/falsch, Prozent = Konfidenz)

In [ ]:
test_df = pd.read_csv(SPLITS / "test.csv")
samples = test_df.sample(3)

fig, axes = plt.subplots(1, 3, figsize=(13, 6))

for ax, (_, row) in zip(axes, samples.iterrows()):
    image = Image.open(row["path"]).convert("RGB")
    ax.imshow(image)
    ax.axis("off")

    lines = [f"Wahrheit: {row['class_name']}", ""]
    for name, model in models.items():
        (top1, p), *_ = predict_top3(model, image)
        mark = "\u2713" if top1 == row["class_name"] else "\u2717"
        lines.append(f"{mark} {name}: {top1} ({p:.0%})")

    ax.set_title("\n".join(lines), fontsize=9, loc="left", family="monospace")

plt.tight_layout()
plt.show()

## Teil 2: Eigene Fotos (Domain Shift)

Die Modelle kennen nur 64×64-**Satellitenbilder** aus 10 Landnutzungsklassen.
Ein normales Foto vom Boden ist weit außerhalb der Trainingsverteilung —
trotzdem *müssen* die Modelle eine der 10 Klassen wählen.
Spannend für die Diskussion: Wie sicher sind sie sich dabei?
(Hohe Konfidenz bei unsinnigem Input = klassisches Problem neuronaler Netze.)

In [ ]:
own_paths = sorted(
    p for p in Path("demo_images").glob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
assert own_paths, "no images found in demo_images/"

fig, axes = plt.subplots(1, len(own_paths), figsize=(6.5 * len(own_paths), 7))
if len(own_paths) == 1:
    axes = [axes]

for ax, path in zip(axes, own_paths):
    image = Image.open(path).convert("RGB")
    ax.imshow(image)
    ax.axis("off")

    lines = [path.name, ""]
    for name, model in models.items():
        top3 = predict_top3(model, image)
        top3_str = ", ".join(f"{c} {p:.0%}" for c, p in top3)
        lines.append(f"{name}:")
        lines.append(f"   {top3_str}")

    ax.set_title("\n".join(lines), fontsize=9, loc="left", family="monospace")

plt.tight_layout()
plt.show()

## Teil 3: Warum entscheiden die Modelle so? (Interpretability)

Die Methoden kommen aus `scripts/interpretability.py`, hier live auf beliebige Bilder angewendet:

- **ConvNeXt (CNN) → Grad-CAM:** Die Gradienten der vorhergesagten Klasse gewichten die
  Feature-Maps der letzten Conv-Stufe — zeigt, *welche Bildbereiche* die Entscheidung getrieben haben.
- **DeiT (Transformer) → Attention Rollout:** Die Attention-Matrizen aller Layer werden
  aufmultipliziert — zeigt, *worauf das Klassifikations-Token über das ganze Netz hinweg schaut*.

Rot/Gelb = wichtig für die Entscheidung, Blau = kaum Einfluss.

In [ ]:
import numpy as np

from scripts.interpretability import gradcam, attention_rollout, load_image


def show_heatmaps(path, true_class=None):
    """original + heatmap per model, title shows the prediction (✓/✗ as above)"""
    tensor, rgb = load_image(Path(path))
    tensor = tensor.to(DEVICE)

    fig, axes = plt.subplots(1, 4, figsize=(14, 4.2))
    axes[0].imshow(rgb)
    axes[0].set_title(f"Original — {true_class if true_class else Path(path).name}", fontsize=9)

    for ax, (name, model) in zip(axes[1:], models.items()):
        # transformer (has .blocks) -> attention rollout, cnn -> grad-cam
        if hasattr(model, "blocks"):
            heatmap, predicted = attention_rollout(model, tensor)
            method = "Attention Rollout"
        else:
            heatmap, predicted = gradcam(model, tensor)
            method = "Grad-CAM"

        overlay = np.clip(0.5 * rgb + 0.5 * plt.get_cmap("jet")(heatmap)[..., :3], 0, 1)
        ax.imshow(overlay)

        mark = ""
        if true_class is not None:
            mark = "\u2713 " if classes[predicted] == true_class else "\u2717 "
        ax.set_title(f"{name} ({method})\n{mark}{classes[predicted]}", fontsize=9)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# random test image, re-running the cell draws a new one
row = test_df.sample(1).iloc[0]
show_heatmaps(row["path"], row["class_name"])

In [ ]:
# own photos (domain shift): what do the models look at when the
# image fits none of the 10 classes? re-running draws new photos
for p in random.sample(own_paths, min(2, len(own_paths))):
    show_heatmaps(p)

### Spickzettel für die Vorstellung

1. **Teil 1** zeigt: Alle drei Modelle sind auf EuroSAT nahezu perfekt (~98–99% Test-Accuracy) —
   Unterschiede sieht man eher bei den Konfidenzen als bei den Vorhersagen.
2. Zelle mehrfach ausführen, bis ein Bild dabei ist, wo sich die Modelle uneinig sind —
   das sind typischerweise verwandte Klassen (z.B. PermanentCrop vs. HerbaceousVegetation).
3. **Teil 2** ist der Kontrast: außerhalb der Trainingsverteilung sind die Vorhersagen
   bedeutungslos, die Konfidenz aber oft trotzdem hoch → Überleitung zur Robustheits-Analyse (Stufe 3).
4. **Teil 3** beantwortet das Warum hinter der Vorhersage: Grad-CAM (ConvNeXt) vs.
   Attention Rollout (DeiT) — dieselben Methoden wie in der Interpretability-Analyse
   (`scripts/interpretability.py`, Beispiele unter `results/figures/interpretability/`).